In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
from rasterio.plot import show
import numpy as np
from rasterio.crs import CRS

# 0.9 - Merge RegBL and Minergie

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `18_MINERGIE_buildings`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Minergie

In [ ]:
df_minergie = pd.read_csv(f'{ROOT}data/processed/02_MINERGIE_buildings.csv', index_col=0)

df_minergie.head()

### Current dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.5.csv', index_col=0)

df_regbl.head()

In [ ]:
df_regbl.shape

## Convert datasets to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

df_minergie = gpd.GeoDataFrame(df_minergie, geometry=gpd.points_from_xy(df_minergie["Coordonnée E"], df_minergie["Coordonnée N"]), crs='EPSG:2056')

## Perform spatial joint

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
df_regbl.plot(ax=ax, color='blue', alpha=0.5)
df_minergie.plot(ax=ax, color='red', markersize=10, alpha=0.5)
plt.show()

In [ ]:
df_regbl.shape

In [ ]:
df_regbl = gpd.sjoin(df_regbl, df_minergie, how="left", predicate="within")

In [ ]:
df_regbl.shape

In [ ]:
df_regbl['Miner - Standard de minergie'].isna().sum()

In [ ]:
296958 - 296822

Only 136 samples of our 296958 long dataset are minergies. This is approximately 0.45%, which is coherent since approximately 1% of the full RegBL dataset are minergies and radon measurements started before minergies existed.

## Drop Minergie coordinates

In [ ]:
df_regbl = df_regbl.drop(columns=['Coordonnée E_right', 'Coordonnée N_right'])

## Rename coordinates properly

In [ ]:
# Rename coordinates as before (Coordonnée E and Coordonnée N)
df_regbl = df_regbl.rename(columns={'Coordonnée E_left': 'Coordonnée E', 'Coordonnée N_left': 'Coordonnée N'})

## Convert current dataset back to DataFrame

In [ ]:
df_regbl = pd.DataFrame(df_regbl.drop(columns='geometry'))

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.6.csv")